# Margin after quantization (RDR-023)

Rebuilds the F16 and Q3_K_M GGUFs of seven existing runs from their LoRA adapters
(no training), checks every rebuilt file is byte-identical to the original
(SHA-256 in the results ledger), and records first-token margins with
`scripts/margin_quant.py`. Output: `margins_quant.zip`.

**Before you start:** Accelerator **GPU T4 x2**, Internet **On**, and add the
dataset made from `margin_quant_inputs.zip` (Add Input). Run all cells top to bottom.


In [ ]:
# 1. SETTINGS
REPO_URL  = "https://github.com/NIghtIngale340/sturdy-broccoli.git"
BRANCH    = "main"
INPUT_DIR = "/kaggle/input"   # searched recursively for the adapter folders

B3, B15 = "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"
RUNS = [  # (prefix in the ledger, adapter folder name, base model, scale)
    ("dose3b_k25_s42",  "dose3b_k25_s42",  B3,  "3B"),
    ("dose3b_k30_s42",  "dose3b_k30_s42",  B3,  "3B"),
    ("dose3b_k40_s42",  "dose3b_k40_s42",  B3,  "3B"),
    ("dose3b_k100_s42", "dose3b_k100_s42", B3,  "3B"),
    ("dose_k30_s123",   "dose_k30_s123",   B15, "1.5B"),
    ("dose_k35_s123",   "dose_k35_s123",   B15, "1.5B"),
    ("cloud_sat_s123",  "saturated_s123",  B15, "1.5B"),
]


In [ ]:
# 2. Where things live (big files go to non-persisted disk)
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules
WORK = "/tmp/work" if ON_KAGGLE else "/content/work"
OUT  = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/slm_research"
os.makedirs(WORK, exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle gives 2 T4s; device_map="auto" would split the model
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
NOTES = f"{'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'cloud'} {gpu.split(',')[0]}"
print("GPU:", gpu, "| notes tag:", NOTES)

def sh(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, stop the notebook on failure."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
# 3. Get the code. Private repo? Add a secret named GITHUB_TOKEN
#    (Kaggle: Add-ons → Secrets; Colab: the key icon on the left).
token = ""
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    elif ON_COLAB:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # public repo: no token needed
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.exists(REPO):
    sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")
sh("git log -1 --format='repo at %h %s'", cwd=REPO)

In [ ]:
# 4. Pinned Python packages (requirements.txt). ~5 min.
sh("pip uninstall -y -q torchvision torchaudio torchao")   # Kaggle extras that clash with the pinned torch/peft; local env has none
sh("pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124")
sh("pip install -q -r requirements.txt", cwd=REPO)

In [ ]:
# 5. llama.cpp at the pinned commit, built with CUDA. ~10-15 min.
LLAMA_COMMIT = "b49650adb31f2e49a0d76113aeb1792134fd8413"
L = f"{REPO}/llama.cpp"
if not os.path.exists(f"{L}/build/bin/llama-server"):
    if not os.path.exists(L):
        sh(f"git clone https://github.com/ggml-org/llama.cpp {L}")
    sh(f"git checkout -q {LLAMA_COMMIT}", cwd=L)
    sh("rm -rf build", cwd=L)
    # 75 = T4 ("native" detection is flaky on Kaggle). NO_VMM: Kaggle has no libcuda.so to link;
    # it only changes GPU memory pooling, not results.
    sh("cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
       "-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc -DGGML_CUDA_NO_VMM=ON", cwd=L)
    sh(f"cmake --build build --config Release -j{os.cpu_count()} --target llama-quantize llama-server", cwd=L)
sh("git rev-parse HEAD", cwd=L)

In [ ]:
# 6. Same test data as the local runs (guardrail; RDR-015 requirement 4)
import hashlib, json
want = json.load(open(f"{REPO}/results/seeds_1p5b_provenance.json"))["data/splits/main_test.json"]["sha256"]
got = hashlib.sha256(open(f"{REPO}/data/splits/main_test.json", "rb").read()).hexdigest()
assert got == want, f"main_test.json hash mismatch: {got}"
print("main_test.json OK", got[:16])

In [ ]:
# 7. Rebuild, verify bytes, measure margins. ~10 min per 3B run, ~5 per 1.5B run.
#    Safe to re-run after a disconnect: finished runs are skipped.
import glob, shutil
ledger = {r["exp_id"]: r for r in map(json.loads, open(f"{REPO}/results/master_results.jsonl"))}
SHA = {}
for prefix, adapter, base, scale in RUNS:
    if os.path.exists(f"{REPO}/results/margins_quant/{prefix}_Q3_K_M.jsonl"):
        print("done already:", prefix); continue
    src = [p for p in glob.glob(f"{INPUT_DIR}/**/{adapter}/adapter_config.json", recursive=True)]
    assert src, f"adapter {adapter} not found under {INPUT_DIR}"
    dst = f"{REPO}/models/lora_adapters/{prefix}"
    sh(f"rm -rf {dst} && mkdir -p {dst} && cp -r {os.path.dirname(src[0])}/. {dst}/")
    sh(f"python3 scripts/04_merge_checkpoint.py --base-model {base} --adapter-dir {dst} "
       f"--output-dir models/merged_fp16/{prefix}", cwd=REPO)
    sh(f"python3 scripts/05_quantize_gguf.py --merged-dir models/merged_fp16/{prefix} "
       f"--prefix {prefix} --ladder F16 Q3_K_M", cwd=REPO)
    for q in ("F16", "Q3_K_M"):
        g = f"{REPO}/models/gguf/{prefix}_{q}.gguf"
        h = hashlib.sha256()
        with open(g, "rb") as f:
            for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
        want = ledger[f"EXP-{scale}_{prefix}_{q}"]["gguf_sha256"]
        SHA[f"{prefix}_{q}"] = {"rebuilt": h.hexdigest(), "ledger": want}
        print(f"{prefix}_{q}: {'IDENTICAL' if h.hexdigest() == want else 'DIFFERENT'} to the ledger")
        sh(f"python3 scripts/margin_quant.py {prefix} {q}", cwd=REPO)
    sh(f"rm -rf models/merged_fp16/{prefix} models/gguf/{prefix}_*.gguf", cwd=REPO)
    os.makedirs(f"{REPO}/results/margins_quant", exist_ok=True)
    old = json.load(open(p)) if os.path.exists(p := f"{REPO}/results/margins_quant/rebuild_sha256.json") else {}
    json.dump({**old, **SHA}, open(p, "w"), indent=1)


In [ ]:
# 8. Pack for download
import zipfile
zpath = f"{OUT}/margins_quant.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{REPO}/results/margins_quant/*"):
        z.write(f, os.path.relpath(f, REPO))
print(open(f"{REPO}/results/margins_quant/rebuild_sha256.json").read())
print("download:", zpath)
